# Instructions :)
1. Runtime → Change runtime type → T4 GPU
2. Download all-data.csv from Kaggle (Sentiment Analysis for Financial News) and upload it when asked
3. Run the cells in order

In [1]:
import torch
print(torch.cuda.is_available())   # this should print true!

True


In [ ]:
!pip install transformers datasets accelerate scikit-learn

In [ ]:
import os
if not os.path.exists("all-data.csv"):
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    if name != "all-data.csv":
        os.rename(name, "all-data.csv")
print(os.listdir("."))

# Click Choose Files when the button appears, and select the all data.csv from Kaggle!

## Fine-tune and score
The next cell splits the data (same split as my baseline models, so the 970 test sentences are identical), fine-tunes FinBERT for 3 epochs, and prints the report card.

In [ ]:
import os, torch, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)

# 1. Load the data
df = pd.read_csv("all-data.csv", encoding="latin-1", names=["sentiment", "text"])

# 2. Same split as before, so the 970 test sentences are the same ones
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["sentiment"],
    test_size=0.2, random_state=42, stratify=df["sentiment"]
)

# 3. Load FinBERT
name = "ProsusAI/finbert"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)
label2id = {k.lower(): v for k, v in model.config.label2id.items()}
id2label = {v: k for k, v in label2id.items()}

# 4. Turn sentences into numbers the model can read
class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(texts.tolist(), truncation=True, padding=True, max_length=128)
        self.labels = [label2id[l] for l in labels]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

train_ds = DS(X_train, y_train)
test_ds = DS(X_test, y_test)

# 5. Training settings
args = TrainingArguments(
    output_dir="finbert-tuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_strategy="no",
    report_to="none",
)

# 6. Train, then take the test
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()

out = trainer.predict(test_ds)
pred_ft = [id2label[i] for i in out.predictions.argmax(-1)]
print("Accuracy:", (pd.Series(pred_ft, index=y_test.index) == y_test).mean())
print(classification_report(y_test, pred_ft))

In [ ]:
trainer.save_model("my_finbert")
tok.save_pretrained("my_finbert")
!zip -r my_finbert.zip my_finbert

from google.colab import files
files.download("my_finbert.zip")
